# テキスト系特徴量実験（train_baseline 複製）

- **目的**: **movie_info**（あらすじ）・**movie_title** をガッツリ扱うテキスト特徴量を追加したときのベースライン（34特徴）からの伸びを比較する。
- **ベース**: train_baseline と同じ 34 特徴（3C3 含む）。各実験は「ベース ＋ テキスト由来の追加」のみ。
- **検証**: 時系列CV（2013〜2016 を val）。TF-IDF 等は **各 fold の tr のみ** で fit し val/test に transform して適用（リーク防止）。
- **実験例**: movie_info の長さ・語数、movie_info の TF-IDF（top-K または SVD で次元圧縮）、movie_title の長さ・語数、組み合わせ。

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

import sys, os
if os.path.basename(os.getcwd()) == "終わった実験環境":
    sys.path.insert(0, os.path.abspath(".."))
from preprocess import load_train_test
from feature_engineering import create_features
from lib.encodings import movie_info_meta as enc_movie_info_meta
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [3]:
train, test = load_train_test()
print(f"Train: {len(train):,}, Test: {len(test):,}")

Train: 653,507, Test: 40,716


In [4]:
train = create_features(train)
test = create_features(test)

# モデルにそのまま使う列のうち、object は category に（欠損は "missing"）
for col in ["rotten_tomatoes_link", "critic_name", "movie_title", "movie_info", "directors", "authors", "actors", "production_company"]:
    if col in train.columns and col in test.columns:
        train[col] = train[col].fillna("missing").astype("category")
        test[col] = test[col].fillna("missing").astype("category")
# 数値で欠損がある列は train の中央値で埋める
for col in ["runtime"]:
    if col in train.columns and col in test.columns and train[col].isna().any():
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col] = test[col].fillna(med)

print("特徴量作成完了.")

特徴量作成完了.


In [5]:
# 3C3 パターン: 時系列TE（critic_name, production_company）＋ TE 3ビン ＋ runtime_bin, movie_age_bin, release_decade
def _ts_te_col_full(df_tr, df_te, col, target_name="target", m=10):
    """時系列TE: tr は「その行より前」のみで平均、te は tr のカテゴリ別スムージング平均でマッピング。"""
    global_mean = float(df_tr[target_name].mean())
    tr_s = df_tr.sort_values("review_date")
    g = tr_s.groupby(col)[target_name]
    past_sum = g.cumsum() - tr_s[target_name]
    past_count = g.cumcount()
    te_tr = np.where(past_count > 0, (past_sum + m * global_mean) / (past_count + m), global_mean)
    ser_tr = pd.Series(te_tr, index=tr_s.index)
    agg = df_tr.groupby(col)[target_name].agg(["mean", "count"])
    agg["smooth"] = (agg["count"] * agg["mean"] + m * global_mean) / (agg["count"] + m)
    map_ = agg["smooth"].to_dict()
    te_arr = df_te[col].astype(str).map(map_).fillna(global_mean).values if df_te is not None and len(df_te) else np.array([])
    return ser_tr, te_arr

for col in ["critic_name", "production_company"]:
    if col not in train.columns:
        continue
    st, ta = _ts_te_col_full(train, test, col, "target", m=10)
    train[f"{col}_te_ts"] = st.reindex(train.index).fillna(train["target"].mean())
    test[f"{col}_te_ts"] = ta

for c in ["critic_name_te_ts", "production_company_te_ts"]:
    if c not in train.columns:
        continue
    train[c + "_bin"] = pd.cut(train[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)
    test[c + "_bin"] = pd.cut(test[c], bins=[0, 0.33, 0.67, 1.01], labels=[0, 1, 2]).astype(float).fillna(1)

train["runtime_bin"] = pd.cut(train["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
test["runtime_bin"] = pd.cut(test["runtime"], bins=[0, 90, 120, 150, 1000], labels=[0, 1, 2, 3]).astype(float).fillna(1)
def _movie_age_bin(x):
    if pd.isna(x) or x < 0: return 0
    if x < 365: return 1
    if x < 365 * 5: return 2
    if x < 365 * 20: return 3
    return 4
train["movie_age_bin"] = train["movie_age_days"].apply(_movie_age_bin)
test["movie_age_bin"] = test["movie_age_days"].apply(_movie_age_bin)
train["release_decade"] = (train["release_year"] // 10 * 10).fillna(1990)
test["release_decade"] = (test["release_year"] // 10 * 10).fillna(1990)
print("3C3 特徴量追加完了.")

3C3 特徴量追加完了.


In [6]:
# ベース特徴量（train_baseline と同じ 34 列）。実験は「ベース＋1つ」のみ。
FEATURES_BASE = [
    "rotten_tomatoes_link", "critic_name", "top_critic", "publisher_name",
    "movie_title", "movie_info", "content_rating", "directors", "authors", "actors",
    "runtime", "production_company",
    "review_year", "review_month", "review_dayofweek",
    "release_year", "release_month", "release_dayofweek", "movie_age_days",
    "genre_Drama", "genre_Comedy", "genre_Action", "genre_Mystery",
    "genre_Fantasy", "genre_Romance", "genre_Horror", "genre_Documentary",
    "critic_name_te_ts", "production_company_te_ts",
    "critic_name_te_ts_bin", "production_company_te_ts_bin",
    "runtime_bin", "movie_age_bin", "release_decade",
]
print(f"ベース特徴量: {len(FEATURES_BASE)}個")

def build_X_for_experiment(config_name, train_df, test_df, tr_idx, val_idx, y_train):
    """config に応じて X_tr, X_val, X_te と特徴量リストを組み立て。TF-IDF 等は tr のみで fit。"""
    tr_df = train_df.iloc[tr_idx].copy()
    val_df = train_df.iloc[val_idx].copy() if val_idx is not None and len(val_idx) else pd.DataFrame()
    te_df = test_df.copy() if test_df is not None else None
    tr_df["target"] = y_train[tr_idx]
    base = [c for c in FEATURES_BASE if c in tr_df.columns]

    if config_name == "base":
        return tr_df[base].copy(), val_df[base].copy() if len(val_df) else None, te_df[base].copy() if te_df is not None else None, base

    # movie_info_meta: 長さ・語数追加（experiment_encodings 使用）
    if config_name == "movie_info_meta":
        extra = enc_movie_info_meta(tr_df, val_df, te_df)
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    # movie_info TF-IDF（top K 列）
    if config_name == "movie_info_tfidf_50": n_tfidf = 50
    elif config_name == "movie_info_tfidf_100": n_tfidf = 100
    elif config_name == "movie_info_tfidf_200": n_tfidf = 200
    else: n_tfidf = None
    if n_tfidf is not None and "movie_info" in tr_df.columns:
        vec = TfidfVectorizer(max_features=n_tfidf, min_df=2, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
        _mi_tr = tr_df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x))
        X_tr_tf = vec.fit_transform(_mi_tr)
        n_actual = X_tr_tf.shape[1]
        _mi_val = val_df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x)) if len(val_df) else None
        _mi_te = te_df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x)) if te_df is not None and len(te_df) else None
        X_val_tf = vec.transform(_mi_val) if _mi_val is not None else None
        X_te_tf = vec.transform(_mi_te) if _mi_te is not None else None
        for i in range(n_actual):
            tr_df[f"tfidf_mi_{i}"] = np.asarray(X_tr_tf[:, i].todense()).ravel()
            if len(val_df): val_df[f"tfidf_mi_{i}"] = np.asarray(X_val_tf[:, i].todense()).ravel()
            if te_df is not None and len(te_df): te_df[f"tfidf_mi_{i}"] = np.asarray(X_te_tf[:, i].todense()).ravel()
        extra = [f"tfidf_mi_{i}" for i in range(n_actual)]
        feats = base + extra
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    # movie_info TF-IDF → SVD で次元圧縮
    if config_name == "movie_info_tfidf_svd20" and "movie_info" in tr_df.columns:
        vec = TfidfVectorizer(max_features=200, min_df=2, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
        _mi_tr = tr_df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x))
        X_tr_tf = vec.fit_transform(_mi_tr)
        _mi_val = val_df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x)) if len(val_df) else None
        _mi_te = te_df["movie_info"].apply(lambda x: "" if pd.isna(x) else str(x)) if te_df is not None and len(te_df) else None
        X_val_tf = vec.transform(_mi_val) if _mi_val is not None else None
        X_te_tf = vec.transform(_mi_te) if _mi_te is not None else None
        svd = TruncatedSVD(n_components=20, random_state=42)
        tr_svd = svd.fit_transform(X_tr_tf)
        for i in range(20):
            tr_df[f"svd_mi_{i}"] = tr_svd[:, i]
            if len(val_df): val_df[f"svd_mi_{i}"] = svd.transform(X_val_tf)[:, i]
            if te_df is not None and len(te_df): te_df[f"svd_mi_{i}"] = svd.transform(X_te_tf)[:, i]
        extra = [f"svd_mi_{i}" for i in range(20)]
        feats = base + extra
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    # movie_title 長さ・語数
    if config_name == "movie_title_len" and "movie_title" in tr_df.columns:
        tr_df["movie_title_len"] = tr_df["movie_title"].astype(str).str.len()
        if len(val_df): val_df["movie_title_len"] = val_df["movie_title"].astype(str).str.len()
        if te_df is not None and len(te_df): te_df["movie_title_len"] = te_df["movie_title"].astype(str).str.len()
        feats = base + ["movie_title_len"]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats
    if config_name == "movie_title_word_count" and "movie_title" in tr_df.columns:
        tr_df["movie_title_word_count"] = tr_df["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
        if len(val_df): val_df["movie_title_word_count"] = val_df["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
        if te_df is not None and len(te_df): te_df["movie_title_word_count"] = te_df["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
        feats = base + ["movie_title_word_count"]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    # movie_info + movie_title のメタ（長さ・語数）
    if config_name == "movie_info_and_title_meta":
        extra = enc_movie_info_meta(tr_df, val_df, te_df)
        if "movie_title" in tr_df.columns:
            tr_df["movie_title_len"] = tr_df["movie_title"].astype(str).str.len()
            tr_df["movie_title_word_count"] = tr_df["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
            if len(val_df): val_df["movie_title_len"] = val_df["movie_title"].astype(str).str.len(); val_df["movie_title_word_count"] = val_df["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
            if te_df is not None and len(te_df): te_df["movie_title_len"] = te_df["movie_title"].astype(str).str.len(); te_df["movie_title_word_count"] = te_df["movie_title"].astype(str).str.split().str.len().fillna(0).astype(int)
            extra = extra + ["movie_title_len", "movie_title_word_count"]
        feats = base + [f for f in extra if f in tr_df.columns]
        return tr_df[feats].copy(), val_df[feats].copy() if len(val_df) else None, te_df[feats].copy() if te_df is not None else None, feats

    return tr_df[base].copy(), val_df[base].copy() if len(val_df) else None, te_df[base].copy() if te_df is not None else None, base

# 実施済み: base, movie_info_meta, movie_info_tfidf_50/100/200 → §16.6 に結果記録。続きのみ実行用。
EXPERIMENT_CONFIGS = [
    "base",                         # 基準（再実行で delta 計算用）
    "movie_info_tfidf_svd20",       # TF-IDF → SVD 20 次元
    "movie_title_len",              # タイトル文字数
    "movie_title_word_count",      # タイトル語数
    "movie_info_and_title_meta",    # 両方の長さ・語数
]
print("実験設定（テキスト系）:", len(EXPERIMENT_CONFIGS), "件")

ベース特徴量: 34個
実験設定（テキスト系）: 5 件


### 各実験設定の違い（EXPERIMENT_CONFIGS）

| 設定名 | ベースに足すもの | 追加列数 |
|--------|------------------|----------|
| **base** | なし | 0（34 特徴のみ） |
| **movie_info_meta** | movie_info の**文字数・語数**（experiment_encodings） | +2 |
| **movie_info_tfidf_50** | movie_info の **TF-IDF 上位 50 列**（1-gram＋2-gram） | +50 程度 |
| **movie_info_tfidf_100** | 同上 **100 列** | +100 程度 |
| **movie_info_tfidf_200** | 同上 **200 列** | +200 程度 |
| **movie_info_tfidf_svd20** | movie_info の TF-IDF(200) を **SVD で 20 次元に圧縮** | +20 |
| **movie_title_len** | **タイトルの文字数** 1 列 | +1 |
| **movie_title_word_count** | **タイトルの語数** 1 列 | +1 |
| **movie_info_and_title_meta** | movie_info の長さ・語数 **＋** タイトルの長さ・語数 | +4 |

- **TF-IDF 系**（50/100/200/svd20）は「単語の重要度」でベクトル化。**svd20** だけ次元を 20 にまとめてノイズを減らしている。
- **meta 系**は「長さ・語数」だけなので、テキストの中身は使わず粗い情報だけ追加。
- いずれも **movie_info / movie_title の category は残したまま** 追加している（置き換えではない）。

### TF-IDF が効かなかった場合に試せること

- **テキストをやめてメタだけ**: movie_info は **長さ・語数のみ**（movie_info_meta）に寄せる。既に実験済み。
- **Bag of Words（CountVectorizer）**: TF だけで IDF なし。語の「出た回数」だけ。スパースになりやすいが、TF-IDF と逆の性質なので試す価値あり。
- **パラメータ変更**: `min_df` を大きくしてレア語を捨てる、`ngram_range=(1,1)` で 1-gram のみ、`max_features` を小さくする（例: 20〜30）。
- **事前学習 embedding**: sentence-transformers などで文を固定次元ベクトルにし、その列を追加。計算コストは増えるが、意味的な類似度が効く場合がある。
- **トピックモデル（LDA / NMF）**: テキストを「トピックの混合比」に変換し、その比を数列として追加。
- **movie_info を外す**: テキスト由来を一切使わず、ベース 34 特徴のみで提出する選択肢もある（§10・§13 では長さ・語数は微増〜微減で一貫していない）。

In [7]:
train

,ID,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_date,movie_title,movie_info,content_rating,genres,...,genre_Romance,genre_Horror,genre_Documentary,critic_name_te_ts,production_company_te_ts,critic_name_te_ts_bin,production_company_te_ts_bin,runtime_bin,movie_age_bin,release_decade
0,0,m/0814255,Andrew L. Urban,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.803451,0.489462,2.0,1.0,1.0,0,2010.0
1,1,m/0814255,Louise Keller,False,Urban Cinefile,2010-02-06,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.820255,0.489491,2.0,1.0,1.0,0,2010.0
2,2,m/0814255,David Germain,True,Associated Press,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.413455,0.489582,1.0,1.0,1.0,0,2010.0
3,3,m/0814255,Nick Schager,False,Slant Magazine,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.360928,0.489637,1.0,1.0,1.0,0,2010.0
4,4,m/0814255,Bill Goodykoontz,True,Arizona Republic,2010-02-10,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",PG,"Action & Adventure, Comedy, Drama, Science Fic...",...,0,0,0,0.682583,0.489549,2.0,1.0,1.0,0,2010.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653502,653502,m/zulu_dawn,Brandon Judell,False,PopcornQ,2005-08-14,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.702341,0.562785,2.0,1.0,2.0,4,1970.0
653503,653503,m/zulu_dawn,Cole Smithey,False,ColeSmithey.com,2005-11-01,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.820266,0.599219,2.0,1.0,2.0,4,1970.0
653504,653504,m/zulu_dawn,Ken Hanke,False,"Mountain Xpress (Asheville, NC)",2007-03-07,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.711286,0.630048,2.0,1.0,2.0,4,1970.0
653505,653505,m/zulu_dawn,Dennis Schwartz,False,Dennis Schwartz Movie Reviews,2010-09-16,Zulu Dawn,Sir Henry Bartle Frere's (John Mills) vastly o...,PG,"Action & Adventure, Art House & International,...",...,0,0,0,0.602209,0.656474,1.0,1.0,2.0,4,1970.0


In [8]:
# 時系列CVの分割は次のセルで実行

In [9]:
# 時系列CV: 検証年ごとに「その年より前で学習 → その年で検証」
# train の review_year 範囲に合わせて検証年を設定（test が 2017〜 なので 2013〜2016 などで検証）
VAL_YEARS = [2013, 2014, 2015, 2016]
time_splits = []
for vy in VAL_YEARS:
    tr_idx = np.where(train["review_year"] < vy)[0]
    val_idx = np.where(train["review_year"] == vy)[0]
    if len(val_idx) > 0:
        time_splits.append((tr_idx, val_idx))

print(f"時系列CV: {len(time_splits)} folds (val years = {VAL_YEARS})")
for i, (_, val_idx) in enumerate(time_splits):
    print(f"  Fold{i+1}: val_year={VAL_YEARS[i]}, val_n={len(val_idx):,}")

時系列CV: 4 folds (val years = [2013, 2014, 2015, 2016])
  Fold1: val_year=2013, val_n=47,263
  Fold2: val_year=2014, val_n=45,165
  Fold3: val_year=2015, val_n=49,842
  Fold4: val_year=2016, val_n=36,455


In [10]:
# 各設定で時系列CVを実行し、ベースラインからの伸び幅を記録
y = train["target"].values
lgb_params = {
    "objective": "binary", "metric": "auc", "boosting_type": "gbdt",
    "n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
    "random_state": 42, "verbosity": -1,
}

results = []
for config_name in EXPERIMENT_CONFIGS:
    print(f"  {config_name}...", end=" ")
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(time_splits):
        X_tr, X_val, _, feats = build_X_for_experiment(config_name, train, test, tr_idx, val_idx, y)
        if X_val is None or len(X_val) == 0:
            continue
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(20, verbose=False)])
        auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
        fold_scores.append(auc)
    mean_auc = np.mean(fold_scores) if fold_scores else np.nan
    n_feat = len(feats)
    results.append({"config": config_name, "n_feat": n_feat, "CV_AUC": mean_auc})
    print(f"n_feat={n_feat}, AUC={mean_auc:.4f}")

res_df = pd.DataFrame(results)
base_auc = res_df.loc[res_df["config"] == "base", "CV_AUC"].iloc[0]
res_df["delta_from_base"] = res_df["CV_AUC"] - base_auc
res_df = res_df.sort_values("CV_AUC", ascending=False)
print(res_df.to_string(index=False))
print(f"\nベース (base) CV AUC: {base_auc:.4f}")
display(res_df)

  base... n_feat=34, AUC=0.7600
  movie_info_tfidf_svd20... n_feat=54, AUC=0.7601
  movie_title_len... n_feat=35, AUC=0.7600
  movie_title_word_count... n_feat=35, AUC=0.7600
  movie_info_and_title_meta... n_feat=38, AUC=0.7602
                   config  n_feat   CV_AUC  delta_from_base
movie_info_and_title_meta      38 0.760220         0.000257
   movie_info_tfidf_svd20      54 0.760124         0.000161
                     base      34 0.759963         0.000000
          movie_title_len      35 0.759963         0.000000
   movie_title_word_count      35 0.759963         0.000000

ベース (base) CV AUC: 0.7600


,config,n_feat,CV_AUC,delta_from_base
4,movie_info_and_title_meta,38,0.760220,0.000257
1,movie_info_tfidf_svd20,54,0.760124,0.000161
0,base,34,0.759963,0.000000
2,movie_title_len,35,0.759963,0.000000
3,movie_title_word_count,35,0.759963,0.000000


In [11]:
# 提出用: ベースで全 train を学習して test を予測（必要なら config を変更可）
X_train, _, X_test, feats = build_X_for_experiment("movie_info_and_title_meta", train, test, np.arange(len(train)), np.array([]), y)
model_full = lgb.LGBMClassifier(**lgb_params)
model_full.fit(X_train, y)
final_pred = model_full.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"ID": test["ID"], "target": final_pred})
submission.to_csv(" movie_info_and_title_meta.csv", index=False)
print(f"Saved submission.csv (rows: {len(submission):,}) [ベースで全train学習]")

importance_df = pd.DataFrame({
    "feature": feats,
    "importance": model_full.feature_importances_,
}).sort_values("importance", ascending=False)
display(importance_df)
print(f"\n重要度 Top5: {list(importance_df.head(5)['feature'].values)}")

Saved submission.csv (rows: 40,716) [ベースで全train学習]


,feature,importance
7,directors,754
8,authors,550
0,rotten_tomatoes_link,453
4,movie_title,420
1,critic_name,228
3,publisher_name,172
9,actors,96
11,production_company,86
27,critic_name_te_ts,51
5,movie_info,45



重要度 Top5: ['directors', 'authors', 'rotten_tomatoes_link', 'movie_title', 'critic_name']
